# Training Computer Vision Models

## Loading Database

In [14]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def get_dataloaders(data_dir,
                    train_dir='train',
                    test_dir='test',
                    batch_size=32,
                    image_size=224,
                    mean=None,
                    std=None,
                    num_workers=0,
                    pin_memory=True):
    """
    Returns train and test DataLoaders using ImageFolder layout:
      data_dir/train/<class>/*.jpg
      data_dir/test/<class>/*.jpg

    Defaults to ImageNet normalization if mean/std not provided.
    """
    if mean is None:
        mean = [0.485, 0.456, 0.406]
    if std is None:
        std = [0.229, 0.224, 0.225]

    # Ensure small images are upscaled before cropping to avoid size errors
    train_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    val_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_path = os.path.join(data_dir, train_dir)
    test_path = os.path.join(data_dir, test_dir)

    train_dataset = datasets.ImageFolder(train_path, transform=train_transforms)
    test_dataset = datasets.ImageFolder(test_path, transform=val_transforms)

    train_loader = DataLoader(train_dataset,
                              batch_size=batch_size,
                              shuffle=True,
                              num_workers=num_workers,
                              pin_memory=pin_memory)

    test_loader = DataLoader(test_dataset,
                             batch_size=batch_size,
                             shuffle=False,
                             num_workers=num_workers,
                             pin_memory=pin_memory)

    return train_loader, test_loader, train_dataset.classes

In [15]:
train_loader, val_loader, classes = get_dataloaders('../data/proximal_data/Nutrition_dataset', batch_size=32, image_size=224)

## Optimizing Hyperparemeters

In [3]:
import optuna
import sys
sys.path.append("../")
import importlib
from utils import image_models
importlib.reload(image_models)
import torch
import torch.nn as nn

c:\Users\gnoceras\Documents\HackIL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
# Use existing loaders and class list
subset_fraction = 0.2
num_classes = len(classes)

train_dataset = train_loader.dataset
val_dataset = val_loader.dataset

train_indices = torch.randperm(len(train_dataset))[: int(len(train_dataset) * subset_fraction)]
val_indices = torch.randperm(len(val_dataset))[: int(len(val_dataset) * subset_fraction)]

train_subset_optuna = torch.utils.data.Subset(train_dataset, train_indices)
val_subset_optuna = torch.utils.data.Subset(val_dataset, val_indices)

train_loader_optuna = DataLoader(
    train_subset_optuna,
    batch_size=train_loader.batch_size,
    shuffle=True,
    num_workers=train_loader.num_workers,
    pin_memory=train_loader.pin_memory,
)

val_loader_optuna = DataLoader(
    val_subset_optuna,
    batch_size=val_loader.batch_size,
    shuffle=False,
    num_workers=val_loader.num_workers,
    pin_memory=val_loader.pin_memory,
)

print(f"Number of training batches: {len(train_loader_optuna)}")
print(f"Number of validation batches: {len(val_loader_optuna)}")


def _build_mobilevit(variant: str, num_classes: int):
    # Try direct num_classes support first
    try:
        return image_models.MobileViT(model_name=variant, num_classes=num_classes).to(device)
    except TypeError:
        model = image_models.MobileViT(model_name=variant).to(device)

        # Fallback: replace final classifier layer if present
        if hasattr(model, "classifier"):
            if isinstance(model.classifier, nn.Linear):
                model.classifier = nn.Linear(model.classifier.in_features, num_classes).to(device)
            elif isinstance(model.classifier, nn.Sequential) and isinstance(model.classifier[-1], nn.Linear):
                in_features = model.classifier[-1].in_features
                model.classifier[-1] = nn.Linear(in_features, num_classes).to(device)
        return model


def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    mobilevit_variant = trial.suggest_categorical(
        "mobilevit_variant",
        ["mobilevit_s", "mobilevit_xs", "mobilevit_xxs"]
    )

    model = _build_mobilevit(mobilevit_variant, num_classes)
    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

    num_epochs = 1
    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0

        for x, y in train_loader_optuna:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * x.size(0)

        model.eval()
        running_val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for x, y in val_loader_optuna:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)

                running_val_loss += loss.item() * x.size(0)
                preds = torch.argmax(logits, dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)

        val_loss = running_val_loss / total
        val_acc = correct / total

        scheduler.step()
        trial.report(val_loss, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    trial.set_user_attr("val_acc", val_acc)
    return val_loss

Number of training batches: 134
Number of validation batches: 67


In [6]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=2,
        n_warmup_steps=0
    )
)
study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-03-06 23:30:20,635] A new study created in memory with name: no-name-3be74134-bb3d-40de-8ce1-f515e2cf2064
Best trial: 0. Best value: 0.477428:   5%|▌         | 1/20 [01:24<26:44, 84.44s/it]

[I 2026-03-06 23:31:45,072] Trial 0 finished with value: 0.4774284263451894 and parameters: {'lr': 0.00023665079963576904, 'weight_decay': 1.2453291046421783e-05, 'mobilevit_variant': 'mobilevit_xs'}. Best is trial 0 with value: 0.4774284263451894.


Best trial: 0. Best value: 0.477428:  10%|█         | 2/20 [02:20<20:24, 68.01s/it]

[I 2026-03-06 23:32:41,575] Trial 1 finished with value: 0.5018370997593198 and parameters: {'lr': 0.0008929276566176515, 'weight_decay': 1.7548809812417723e-06, 'mobilevit_variant': 'mobilevit_xs'}. Best is trial 0 with value: 0.4774284263451894.


Best trial: 0. Best value: 0.477428:  15%|█▌        | 3/20 [03:02<15:51, 55.95s/it]

...

[I 2026-03-06 23:47:27,932] Trial 17 pruned. 


Best trial: 16. Best value: 0.247018:  95%|█████████▌| 19/20 [18:06<00:59, 59.25s/it]

[I 2026-03-06 23:48:27,451] Trial 18 pruned. 


Best trial: 16. Best value: 0.247018: 100%|██████████| 20/20 [19:08<00:00, 57.44s/it]

[I 2026-03-06 23:49:29,501] Trial 19 pruned. 


In [7]:
print("Best trial:")
print(study.best_trial.value)

print("Best params:")
for k, v in study.best_trial.params.items():
    print(f"{k}: {v}")

Best trial:
0.24701778752526987
Best params:
lr: 0.0005965725033997052
weight_decay: 0.00017076419163636474
mobilevit_variant: mobilevit_s


Best hyperparameters:

lr: 0.0005965725033997052

weight_decay: 0.00017076419163636474

mobilevit_variant: mobilevit_s

In [8]:
torch.cuda.empty_cache()

## Training the model

In [16]:
num_classes = len(classes)

model = image_models.MobileViT(
    model_name=study.best_trial.params["mobilevit_variant"],
    num_classes=num_classes
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=study.best_trial.params["lr"],
    weight_decay=study.best_trial.params["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50
)

scaler = torch.amp.GradScaler("cuda")

num_epochs = 50
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0
modelName = "MobileViT"

# Track losses for plotting
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_train_loss = 0.0
    train_samples = 0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast("cuda"):
            logits = model(x)
            loss = criterion(logits, y)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_train_loss += loss.item() * x.size(0)
        train_samples += x.size(0)
    
    train_loss = running_train_loss / train_samples
    
    # Validation phase
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            
            with torch.amp.autocast("cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            
            running_val_loss += loss.item() * x.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y).sum().item()
            total += x.size(0)
    
    val_loss = running_val_loss / total
    val_acc = correct / total
    
    # Track losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    scheduler.step()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | Acc: {val_acc:.4f}"
    )

    # Early stopping logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        print(f"Validation loss improved. Saving model at epoch {epoch+1} ✅")
        torch.save(model.state_dict(), f"../models/{modelName}.pth")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epochs. ❌")
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} 🛑")
            break

UnidentifiedImageError: cannot identify image file <_io.BufferedReader name='../data/proximal_data/Nutrition_dataset\\test\\KAB\\0561_4_0.jpg'>

### Visualizing training history

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.title('Fusion Model Training History', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final validation loss: {val_losses[-1]:.6f}")
print(f"Best validation loss: {best_val_loss:.6f}")

## Evaluating Model